In [ ]:
# Import required libraries
from pathlib import Path
from math import ceil

import kornia.augmentation as K

from tree_detection_framework.preprocessing.preprocessing import (
    create_dataloader,
    create_intersection_dataloader,
)
from tree_detection_framework.detection.detector import (
    GeometricTreeTopDetector,
    GeometricTreeCrownDetector,
)
from tree_detection_framework.constants import DATA_FOLDER
from tree_detection_framework.postprocessing.postprocessing import (
    remove_edge_detections,
    suppress_tile_boundary_with_NMS,
    multi_region_NMS,
    single_region_hole_suppression,
)
from tree_detection_framework.preprocessing.preprocessing import visualize_dataloader
from tree_detection_framework.preprocessing.utils import KorniaTransformWrapper

In [ ]:
# The path to a Canopy Height Model raster file
RASTER_FOLDER_PATH = Path(DATA_FOLDER, "emerald-point-chm")
RASTER_FILE_PATH = Path(RASTER_FOLDER_PATH, "chm.tif")
# Prediction file path
OUTPUT_TREETOPS_FILE_PATH = Path(
    RASTER_FOLDER_PATH, "emerald-point-chm-geometric-tree-tops.gpkg"
)
OUTPUT_TREECROWNS_FILE_PATH = Path(
    RASTER_FOLDER_PATH, "emerald-point-chm-geometric-tree-crowns.gpkg"
)
# The size of the chips in pixels
CHIP_SIZE = 512
# The stride between chips in pixels
CHIP_STRIDE = 400
# The spatial resolution that the data is sampled to in meters/pix
RESOLUTION = 0.2
# The blur resolution
BLUR_SIGMA_METERS = 0.5
# The number of tiles to show from the dataloader
N_VIS_TILES = 3

In [ ]:
# Compute the kernel sigma in pixels
kernel_sigma_pixels = BLUR_SIGMA_METERS / RESOLUTION
# Set the kernel size to be odd and over two sigmas, which captures the vast majority of the probability density
kernel_size = 2 * ceil(kernel_sigma_pixels) + 1

# Create the transforms, including a wrapper that keeps singleton batches the same shape
transforms = KorniaTransformWrapper(
    K.AugmentationSequential(
        # Note that sigma is specified as an upper and lower bound. In this case, they are set identically
        K.RandomGaussianBlur(kernel_size=kernel_size, sigma=(kernel_sigma_pixels, kernel_sigma_pixels), p=1.0),
    )
)

# Stage 1: Create a dataloader for the raster data and detect the tree-tops
dataloader = create_dataloader(
    raster_folder_path=RASTER_FOLDER_PATH,
    chip_size=CHIP_SIZE,
    chip_stride=CHIP_STRIDE,
    resolution=RESOLUTION,
    raster_transforms=transforms,
)

In [ ]:
visualize_dataloader(dataloader, n_tiles=N_VIS_TILES)

In [ ]:
treetop_detector = GeometricTreeTopDetector(
    a=0, b=0.0325, c=0.25, confidence_feature="height"
)

treetop_detections = treetop_detector.predict(dataloader)

In [ ]:
# Remove the tree tops that were generated in the edges of tiles. This is an alternative to NMS.
treetop_detections = remove_edge_detections(
    treetop_detections,
    suppression_distance=(CHIP_SIZE - CHIP_STRIDE) * RESOLUTION / 2,
)

In [ ]:
# Optionally, save the treetops to disk
treetop_detections.save(OUTPUT_TREETOPS_FILE_PATH)

In [ ]:
# Stage 2: Combine raster and vector data (from the tree-top detector) to create a new dataloader
raster_vector_dataloader = create_intersection_dataloader(
    raster_data=RASTER_FOLDER_PATH,
    vector_data=treetop_detections,
    chip_size=CHIP_SIZE,
    chip_stride=CHIP_STRIDE,
    resolution=RESOLUTION,
)

treecrown_detector = GeometricTreeCrownDetector(approach="watershed")

treecrown_detections = treecrown_detector.predict(raster_vector_dataloader)

In [ ]:
# Suppress overlapping crown predictions. This step can be slow.
treecrown_detections = multi_region_NMS(
    treecrown_detections, confidence_column="score", intersection_method="IOS"
)

In [ ]:
# Display outputs from the tree crown detector. Note: treetop UIDs get maintained.
treecrown_detections.get_data_frame()

In [ ]:
treecrown_detections.plot(
    raster_file=Path(DATA_FOLDER, "emerald-point-ortho/ortho.tif")
)

In [ ]:
# Post-processing step to remove holes in the tree crowns
treecrown_detections = single_region_hole_suppression(treecrown_detections)
treecrown_detections.plot(
    raster_file=Path(DATA_FOLDER, "emerald-point-ortho/ortho.tif")
)

In [ ]:
treecrown_detections.save(OUTPUT_TREECROWNS_FILE_PATH)